# 03 – E2: CLIP ViT-B/32 đóng băng (Thành viên B)
Chạy **GPU T4**, **Internet = On**. Notebook chỉ: nhập đường dẫn → clone repo → tạo split (cố định theo file đã commit) → chạy `scripts/run_e2_frozen.py`.
**Sketch luôn đọc từ dữ liệu gốc; ảnh thật đọc từ cache 256 px nếu có (`CACHE_ROOT`), theo mục 2.4 của kế hoạch.** Vì vậy phải Add Data cả dataset gốc lẫn dataset cache.
Kết quả: `/kaggle/working/results/E2/metrics.json` + embedding `/kaggle/working/features/E2/*.npy|csv`, nén thành `E2_outputs.zip`, `E2_features.zip`.

In [ ]:
# ======= NHẬP ĐƯỜNG DẪN =======
SKETCHY_RAW = "/kaggle/input/datasets/hongonthi/sketchyextended/sketchy"   # BẮT BUỘC: chứa train/ và val/ (sketch đọc từ đây)
FLICKR_RAW  = "/kaggle/input/datasets/hongonthi/flickr15k/Flickr15K"       # BẮT BUỘC: chứa images/, sketches/, list_images.txt
CACHE_ROOT  = "/kaggle/input/datasets/hongonthi/sbir-cache-256-photos"     # cache CHỈ ẢNH THẬT (chứa sketchy/, flickr15k/); "" = đọc ảnh gốc
REPO        = "https://github.com/thaihoang104569/CS406.git"
SKETCH_DILATE = 0  # 3 = làm dày nét sketch trước khi đưa vào CLIP (thử cho Flickr15K)
# ==============================
import os, sys
for name, path in [("SKETCHY_RAW", SKETCHY_RAW), ("FLICKR_RAW", FLICKR_RAW)] + ([("CACHE_ROOT", CACHE_ROOT)] if CACHE_ROOT else []):
    assert path and os.path.isdir(path), f"{name} không tồn tại: {path!r} - kiểm tra Add Data / đường dẫn"
WORK = "/kaggle/working/CS406"
if os.path.isdir(WORK):
    !git -C {WORK} pull -q
else:
    !git clone -q {REPO} {WORK}
!git -C {WORK} log --oneline -1
SK_PHOTO = f"{CACHE_ROOT}/sketchy" if CACHE_ROOT else SKETCHY_RAW     # ảnh thật Sketchy
FL_PHOTO = f"{CACHE_ROOT}/flickr15k" if CACHE_ROOT else FLICKR_RAW    # ảnh thật Flickr15K
print("sketch:", SKETCHY_RAW, "|", FLICKR_RAW)
print("photo :", SK_PHOTO, "|", FL_PHOTO)

In [ ]:
!pip install -q open_clip_torch
!pip install -q faiss-gpu-cu12 || pip install -q faiss-cpu
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Split (đọc ảnh gốc, giữ đúng 10 lớp val đã commit)

In [ ]:
!python {WORK}/scripts/prepare_data.py --sketchy_root "{SKETCHY_RAW}" --flickr_root "{FLICKR_RAW}" \
    --out_dir /kaggle/working/data_prep --class_split_csv {WORK}/data/splits/sketchy_class_splits.csv \
    --skip_dup_check --no_figures --sample_per_group 50

## 2. Chạy thử nhanh (`--debug 300`, ~1 phút) rồi chạy đầy đủ

In [ ]:
SPL = "/kaggle/working/data_prep/splits"
COMMON = (f'--sketchy_csv {SPL}/sketchy_zero_shot_splits.csv '
          f'--sketchy_root "{SKETCHY_RAW}" --sketchy_photo_root "{SK_PHOTO}" '
          f'--flickr_csv {SPL}/flickr15k_splits.csv '
          f'--flickr_root "{FLICKR_RAW}" --flickr_photo_root "{FL_PHOTO}" '
          f'--sketch_dilate {SKETCH_DILATE} --faiss_gpu --workers 4 --batch_size 256')
!python {WORK}/scripts/run_e2_frozen.py {COMMON} --debug 300 --out_dir /kaggle/working/e2_debug --feat_dir /kaggle/working/e2_debug/features

In [ ]:
!python {WORK}/scripts/run_e2_frozen.py {COMMON} --out_dir /kaggle/working/results/E2 --feat_dir /kaggle/working/features/E2

## 3. Kết quả

In [ ]:
import json, pandas as pd, shutil
m = json.load(open("/kaggle/working/results/E2/metrics.json"))
rows = []
for name, r in m["results"].items():
    rows.append({"set": name, **{f"visual {k}": v for k, v in r["visual"].items()},
                 "text mAP@200": r["text_variant"]["mAP@200"], "text top-1 acc": r["text_variant"]["sketch_top1_accuracy"],
                 **r["timing"]})
display(pd.DataFrame(rows).set_index("set").T)
pc = pd.DataFrame(m["results"]["sketchy_test"]["per_class_visual"]).T.sort_values("mAP@200")
display(pc)
shutil.make_archive("/kaggle/working/E2_outputs", "zip", "/kaggle/working", "results")
shutil.make_archive("/kaggle/working/E2_features", "zip", "/kaggle/working", "features")
!ls -la /kaggle/working/*.zip